# E2b: Unsupervised classifier (labeled cycles only)

Same as E2 but K-means clusters ONLY labeled cycles, not unlabeled.
Tests whether unlabeled data was helping or hurting centroids.

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w','ay_w','az_w']].values
    omega = df[['gx','gy','gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag_deg = np.linalg.norm(omega, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7: return []
    win = int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if win % 2 == 0: win += 1
    win = max(5, min(win, n if n % 2 == 1 else n - 1))
    poly = max(1, min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']), win - 2))
    mag_s = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    mag_s = savgol_filter(mag_s, window_length=win, polyorder=poly, mode='interp')
    peaks, _ = find_peaks(mag_s, distance=max(1, int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)),
                          prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],
                          height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(peaks) == 0: return []
    cycles = []
    for i, p in enumerate(peaks):
        left = 0 if i == 0 else (peaks[i-1] + p) // 2
        right = (len(t)-1) if i == len(peaks)-1 else (p + peaks[i+1]) // 2
        if right > left and (right - left) >= CONFIG['MIN_CYCLE_SAMPLES']:
            cycles.append((left, right))
    return cycles

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T.astype(np.float32)

FINE_MAP = {
    'UR': 'UR', 'ur': 'UR', 'UR0': 'UR', 'UR-CW': 'UR', 'UL0': 'UL',
    'OR': 'OR', 'or': 'OR', 'OR2': 'OR', 'OR3': 'OR', 'OR-OSL': 'OR', 'OR/OSL?': 'OR',
    'OL': 'OL', 'ol': 'OL', 'OL2': 'OL',
    'USR': 'USR', 'USL': 'USL', 'OSR': 'OSR', 'OSL': 'OSL',
    'FB': 'FB', 'fb': 'FB', 'FB2': 'FB', 'BF2': 'BF', 'bf': 'BF',
    'underhand': 'U_generic', 'overhand': 'O_generic', 'dragon_roll': 'DR_generic',
    'sneak_underhand': 'SU_generic', 'sneak_overhand': 'SO_generic',
    'CW': None, 'cw': None, 'CCW': None, 'ccw': None,
    'idle': None, 'Idle3': None, 'Idle-or-ol?': None, 'VQ5': None, 'vq16': None,
}
_FINE_PREFIX = [
    (re.compile(r'^USR$', re.I), 'USR'), (re.compile(r'^USL$', re.I), 'USL'),
    (re.compile(r'^OSR$', re.I), 'OSR'), (re.compile(r'^OSL$', re.I), 'OSL'),
    (re.compile(r'^UR', re.I), 'UR'), (re.compile(r'^UL', re.I), 'UL'),
    (re.compile(r'^OR', re.I), 'OR'), (re.compile(r'^OL', re.I), 'OL'),
    (re.compile(r'^FB', re.I), 'FB'), (re.compile(r'^BF', re.I), 'BF'),
    (re.compile(r'^CW$', re.I), None), (re.compile(r'^CCW$', re.I), None),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
def map_fine_label(raw):
    raw = raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for pat, c in _FINE_PREFIX:
        if pat.match(raw): return c
    return None

COARSE_MAP = {
    'UR': 'underhand', 'UL': 'underhand', 'U_generic': 'underhand',
    'OR': 'overhand', 'OL': 'overhand', 'O_generic': 'overhand',
    'USR': 'sneak_underhand', 'USL': 'sneak_underhand', 'SU_generic': 'sneak_underhand',
    'OSR': 'sneak_overhand', 'OSL': 'sneak_overhand', 'SO_generic': 'sneak_overhand',
    'FB': 'dragon_roll', 'BF': 'dragon_roll', 'DR_generic': 'dragon_roll',
}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
print('Functions defined')

In [ ]:
# ── Load ONLY labeled cycles (no unlabeled) ──────────────────
all_matrices = []; all_fine = []; all_groups = []

def load_cycles(d0p, d1p, name, fine='unlabeled', windows=None):
    df0, df1 = pd.read_csv(d0p), pd.read_csv(d1p)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    cyc0 = detect_cycles(t0, om0, CONFIG['FS'])
    cyc1 = detect_cycles(t1, om1, CONFIG['FS'])
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if windows:
        fp0, fp1 = [], []
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws <= (t0[s0]+t0[e0])/2 < we for ws, we in windows):
                fp0.append((s0,e0)); fp1.append((s1,e1))
        p0, p1 = fp0, fp1
    grp = name.split('/')[0] if '/' in name else name
    cnt = 0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm); all_fine.append(fine); all_groups.append(grp)
        cnt += 1
    return cnt

# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv'))):
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    fine = None
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            if pat in ('underhand','underhand_default'): fine = 'U_generic'
            elif pat == 'overhand': fine = 'O_generic'
            elif pat == 'dragon_roll': fine = 'DR_generic'
            elif pat == 'sneak_underhand': fine = 'SU_generic'
            elif pat == 'sneak_overhand': fine = 'SO_generic'
            break
    if fine is None: continue  # SKIP unlabeled
    n = load_cycles(d0, d1, stem, fine)
    if n > 0: print(f'  {stem:<50s} {fine:<12s} {n:>4d}')

# Heterogeneous (L/R labels)
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f: data = _json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            fine = map_fine_label(seg.get('label', ''))
            if fine is None: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[fine].append((float(s), float(e)))
        for fine, windows in sorted(wbl.items()):
            n = load_cycles(d0, d1, sname + '/' + fine, fine, windows)
            if n > 0: print(f'  {sname}/{fine:<12s} {"":>33s} {n:>4d}')

X_raw = np.array([cm.ravel() for cm in all_matrices])
y_fine = np.array(all_fine)
g_all = np.array(all_groups)

lr_mask = np.array([l in LR_CLASSES for l in y_fine])
X_lr = X_raw[lr_mask]; y_lr = y_fine[lr_mask]; g_lr = g_all[lr_mask]

print('\nTotal labeled cycles: ' + str(len(X_raw)))
print('L/R labeled: ' + str(lr_mask.sum()))
print('Generic labeled: ' + str((~lr_mask).sum()))
for lab, cnt in sorted(Counter(y_fine).items(), key=lambda x: -x[1]):
    print(f'  {lab:<12s}: {cnt:>5d}')

In [ ]:
# ── LOSO: Nearest centroid on LABELED-ONLY clusters ──────────
# For each fold:
#   1. Scale + PCA on labeled train cycles only
#   2. K-means on labeled train only
#   3. Map clusters to dominant L/R label
#   4. Classify test by nearest centroid

K_VALUES = [5, 8, 10, 12, 15]
results = {}

unique_groups = np.unique(g_lr)
class_groups = defaultdict(set)
for label, grp in zip(y_lr, g_lr): class_groups[label].add(grp)
singleton_classes = {c for c, gs in class_groups.items() if len(gs) < 2}
test_groups = [g for g in unique_groups
               if not set(y_lr[g_lr == g]).issubset(singleton_classes)]

for k in K_VALUES:
    all_true, all_pred = [], []
    all_true_c, all_pred_c = [], []

    for g in test_groups:
        te = g_lr == g; tr = ~te
        X_tr, X_te = X_lr[tr], X_lr[te]
        y_tr, y_te = y_lr[tr], y_lr[te]

        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)
        pca = PCA(n_components=min(50, X_tr_s.shape[1], X_tr_s.shape[0]-1), svd_solver='full')
        X_tr_p = pca.fit_transform(X_tr_s)
        X_te_p = pca.transform(X_te_s)

        km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
        tr_clusters = km.fit_predict(X_tr_p)

        # Map clusters
        cmap = {}
        for c in range(k):
            cmask = tr_clusters == c
            if cmask.sum() == 0: cmap[c] = 'unknown'; continue
            cmap[c] = Counter(y_tr[cmask]).most_common(1)[0][0]

        te_clusters = km.predict(X_te_p)
        preds = np.array([cmap.get(c, 'unknown') for c in te_clusters])
        valid = preds != 'unknown'

        all_true.extend(y_te[valid].tolist())
        all_pred.extend(preds[valid].tolist())
        all_true_c.extend([COARSE_MAP.get(l, l) for l in y_te[valid]])
        all_pred_c.extend([COARSE_MAP.get(p, p) for p in preds[valid]])

    all_true = np.array(all_true); all_pred = np.array(all_pred)
    all_true_c = np.array(all_true_c); all_pred_c = np.array(all_pred_c)
    labs_f = sorted(set(all_true) | set(all_pred))
    labs_c = sorted(set(all_true_c) | set(all_pred_c))

    f1f = f1_score(all_true, all_pred, average='macro', labels=labs_f, zero_division=0)
    f1c = f1_score(all_true_c, all_pred_c, average='macro', labels=labs_c, zero_division=0)
    acc = np.mean(all_true == all_pred)

    results[k] = {'f1_fine': f1f, 'f1_coarse': f1c, 'acc': acc,
                   'cm': confusion_matrix(all_true, all_pred, labels=labs_f),
                   'labs': labs_f,
                   'report': classification_report(all_true, all_pred, labels=labs_f, zero_division=0)}
    print(f'k={k:>2d}: fine F1={f1f:.3f}, coarse F1={f1c:.3f}, Acc={acc:.3f}')

In [ ]:
best_k = max(results, key=lambda k: results[k]['f1_fine'])
r = results[best_k]
print(r['report'])

fig, ax = plt.subplots(figsize=(max(7, len(r['labs'])*0.9), max(6, len(r['labs'])*0.8)))
im = ax.imshow(r['cm'], cmap='Blues')
ax.set_xticks(range(len(r['labs']))); ax.set_yticks(range(len(r['labs'])))
ax.set_xticklabels(r['labs'], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(r['labs'], fontsize=8)
for i in range(len(r['labs'])):
    for j in range(len(r['labs'])):
        ax.text(j, i, str(r['cm'][i,j]), ha='center', va='center',
                color='white' if r['cm'][i,j] > r['cm'].max()*0.5 else 'black', fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Nearest Centroid k={best_k} (labeled only, F1={r["f1_fine"]:.3f})')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
print('='*70)
print('E2b: UNSUPERVISED CLASSIFIER (LABELED ONLY) SUMMARY')
print('='*70)
print(f'L/R labeled cycles: {lr_mask.sum()}')
print(f'Generic labeled: {(~lr_mask).sum()}')
print(f'NO unlabeled cycles used in clustering')
print(f'LOSO: re-cluster per fold (no leakage)')
print(f'')
print('Results by k:')
for k in K_VALUES:
    r = results[k]
    print(f'  k={k:>2d}: fine F1={r["f1_fine"]:.3f}, coarse F1={r["f1_coarse"]:.3f}, Acc={r["acc"]:.3f}')
print(f'')
print(f'Best k={best_k}: fine F1={results[best_k]["f1_fine"]:.3f}, coarse F1={results[best_k]["f1_coarse"]:.3f}')
print(f'')
print(f'Comparison:')
print(f'  E2 (with unlabeled, k=15):   fine F1=0.432, coarse F1=0.559')
print(f'  E2b (labeled only, k={best_k}):  fine F1={results[best_k]["f1_fine"]:.3f}, coarse F1={results[best_k]["f1_coarse"]:.3f}')
print(f'  V08 reference (5-class):     F1=0.632')